# Experiment 1: Time and Frequency Domain, and the Pendulum

**Course:** CSAI4014 – Algorithms for Intelligent Systems and Robotics  
**Experiment:** 1

Run every code cell from top to bottom in a fresh kernel. The notebook generates the measurements and figures used in the lab record.

## 1. Aim
To represent a sampled signal in the time domain and in the frequency domain using the discrete Fourier transform, and to simulate the motion of a simple pendulum by numerical integration of its equation of motion.

## 2. Objectives
1. Generate a sampled signal of known composition and state its sampling rate, record length and frequency resolution.
2. Compute an amplitude spectrum and read amplitudes from it in the units of the original signal.
3. Predict and demonstrate aliasing.
4. Write a second-order ordinary differential equation as a first-order system and integrate it numerically.
5. Produce a phase portrait and relate its shape to the energy of the motion.
6. Measure the dependence of a pendulum’s period on its amplitude.

## 3. Requirements
Python 3.10 or later, NumPy, SciPy and Matplotlib. No hardware is required.

## 4. Theory
For uniformly spaced samples, sampling rate is $f_s=1/\Delta t$, record length is $T=N\Delta t$, frequency resolution is $\Delta f=f_s/N=1/T$, and the Nyquist frequency is $f_s/2$. A component above the Nyquist limit folds into a lower apparent frequency (aliasing). For the pendulum, with $\omega=\dot\theta$, the state equations are $\dot\theta=\omega$ and $\dot\omega=-(b/m)\omega-(g/L)\sin\theta$. The small-angle period is $T_0=2\pi\sqrt{L/g}$.

## 5. Part A: Build a signal and read its spectrum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

fs, T = 500.0, 2.0
t = np.arange(0, T, 1/fs)
N = t.size

rng = np.random.default_rng(0)
x = (1.00*np.sin(2*np.pi*5*t)
     + 0.50*np.sin(2*np.pi*20*t)
     + 0.25*np.sin(2*np.pi*50*t)
     + 0.20*rng.standard_normal(N))

X = np.fft.rfft(x)
f = np.fft.rfftfreq(N, d=1/fs)
amp = np.abs(X) * 2.0 / N
amp[0] /= 2.0

print("Sampling frequency =", fs, "Hz")
print("Number of samples =", N)
print("Record length =", T, "s")
print("Frequency resolution =", fs/N, "Hz")
print("Nyquist frequency =", fs/2, "Hz")
for target in [5, 20, 50]:
    idx = np.argmin(np.abs(f-target))
    print(f"Set {target} Hz: measured peak {f[idx]:.2f} Hz, amplitude {amp[idx]:.3f}")

plt.figure(figsize=(9,4))
plt.plot(t, x, lw=0.8)
plt.xlim(0, 0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (signal units)")
plt.title("Constructed signal in the time domain")
plt.figtext(0.5, 0.01, "Figure 1.1: Sum of three sinusoids and seeded noise.", ha="center")
plt.tight_layout(rect=[0,0.06,1,1])
plt.show()

plt.figure(figsize=(9,4))
plt.plot(f, amp)
plt.xlim(0, 70)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude (signal units)")
plt.title("One-sided amplitude spectrum")
plt.figtext(0.5, 0.01, "Figure 1.2: Peaks occur at the three component frequencies.", ha="center")
plt.tight_layout(rect=[0,0.06,1,1])
plt.show()


## 6. Part B: Demonstrate aliasing

In [ ]:
fs_a = 8.0
t_a = np.arange(0, 1.0, 1/fs_a)
x_a = np.sin(2*np.pi*5*t_a)

X_a = np.fft.rfft(x_a)
f_a = np.fft.rfftfreq(t_a.size, d=1/fs_a)
alias_peak = f_a[np.argmax(np.abs(X_a))]
print("Sampling frequency =", fs_a, "Hz")
print("Nyquist frequency =", fs_a/2, "Hz")
print("Measured apparent peak =", alias_peak, "Hz")
print("Expected alias frequency =", abs(fs_a-5), "Hz")

t_dense = np.linspace(0, 1, 1000)
plt.figure(figsize=(9,4))
plt.plot(t_a, x_a, "o", label="Samples at 8 Hz")
plt.plot(t_dense, np.sin(2*np.pi*5*t_dense), label="True 5 Hz signal")
plt.plot(t_dense, np.sin(2*np.pi*3*t_dense), "--", label="Apparent 3 Hz signal")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Aliasing of a 5 Hz signal sampled at 8 Hz")
plt.legend()
plt.figtext(0.5, 0.01, "Figure 1.3: The samples also fit an apparent 3 Hz signal.", ha="center")
plt.tight_layout(rect=[0,0.06,1,1])
plt.show()


## 7. Part C: Pendulum, damping and amplitude effect

In [ ]:
g, L, m = 9.81, 1.0, 1.0

def pendulum(t, y, b):
    theta, omega = y
    return [omega, -(b/m)*omega - (g/L)*np.sin(theta)]

def simulate(theta0_deg, b, T=20.0, n=4000):
    te = np.linspace(0, T, n)
    sol = solve_ivp(
        pendulum, [0, T],
        [np.deg2rad(theta0_deg), 0.0],
        args=(b,), t_eval=te, rtol=1e-9, atol=1e-12
    )
    return sol.t, sol.y[0], sol.y[1]

T0 = 2*np.pi*np.sqrt(L/g)
print("Small-angle predicted period T0 =", round(T0, 4), "s")

plt.figure(figsize=(9,4))
for b, label in [(0.0, "Undamped b=0"), (0.3, "Light damping b=0.3"), (2.0, "Heavy damping b=2.0")]:
    tt, th, om = simulate(40, b, T=20, n=4000)
    plt.plot(tt, np.rad2deg(th), label=label)
plt.xlabel("Time (s)")
plt.ylabel("Angle (degrees)")
plt.title("Pendulum angle against time for different damping coefficients")
plt.legend()
plt.figtext(0.5, 0.01, "Figure 1.4: Damping causes the oscillation amplitude to decay.", ha="center")
plt.tight_layout(rect=[0,0.06,1,1])
plt.show()

plt.figure(figsize=(7,6))
for b, label in [(0.0, "Undamped b=0"), (0.3, "Light damping b=0.3"), (2.0, "Heavy damping b=2.0")]:
    tt, th, om = simulate(40, b, T=20, n=4000)
    plt.plot(np.rad2deg(th), om, label=label)
plt.xlabel("Angle (degrees)")
plt.ylabel("Angular velocity (rad/s)")
plt.title("Pendulum phase portraits")
plt.legend()
plt.figtext(0.5, 0.01, "Figure 1.5: Damped trajectories move inward as energy is dissipated.", ha="center")
plt.tight_layout(rect=[0,0.06,1,1])
plt.show()


### Period measurement

In [ ]:
def period_zero_crossings(tt, th):
    # Interpolate positive-going zero crossings and average their spacing.
    inds = np.where((th[:-1] < 0) & (th[1:] >= 0))[0]
    crossings = []
    for i in inds:
        frac = -th[i] / (th[i+1] - th[i])
        crossings.append(tt[i] + frac*(tt[i+1]-tt[i]))
    return float(np.mean(np.diff(crossings))) if len(crossings) >= 2 else np.nan

def period_spectrum(tt, th):
    dt = tt[1] - tt[0]
    A = np.abs(np.fft.rfft(th))
    fr = np.fft.rfftfreq(th.size, d=dt)
    peak = fr[1:][np.argmax(A[1:])]  # skip zero-frequency bin
    return 1.0/peak

for angle in (5, 95, 175):
    tt, th, om = simulate(angle, 0.0, T=60.0, n=12000)
    print(f"{angle:>3} degrees: zero-crossing period = {period_zero_crossings(tt, th):.3f} s; "
          f"spectral period = {period_spectrum(tt, th):.3f} s")

angles = [5, 25, 45, 65, 85, 95, 115, 135, 155, 165, 175]
measured_periods = []
for angle in angles:
    tt, th, om = simulate(angle, 0.0, T=80.0, n=16000)
    measured_periods.append(period_zero_crossings(tt, th))

plt.figure(figsize=(9,4))
plt.plot(angles, measured_periods, "o-", label="Measured period")
plt.axhline(T0, ls="--", label=f"Small-angle prediction T0={T0:.3f} s")
plt.xlabel("Initial angle (degrees)")
plt.ylabel("Period (s)")
plt.title("Pendulum period versus initial amplitude")
plt.legend()
plt.figtext(0.5, 0.01, "Figure 1.6: The period increases with initial amplitude.", ha="center")
plt.tight_layout(rect=[0,0.06,1,1])
plt.show()


## 8. Results and observations
Record the values printed by your own notebook. In particular, record $f_s$, $N$, $T$, $\Delta f$, $f_s/2$, the measured peak frequencies and amplitudes, the aliased peak frequency, $T_0$, periods measured by zero crossings and spectrum, and the damping coefficients.

## 9. Discussion
The three known sinusoidal components should appear as peaks in the one-sided spectrum. With a sampling rate of 8 Hz, the 5 Hz component is above the 4 Hz Nyquist limit and appears at 3 Hz. For the pendulum, increasing damping reduces oscillation amplitude; the undamped phase portrait is closed while damped trajectories move inward. The period increases for large initial angles, showing the limitation of the small-angle approximation.

## 10. Applications
Spectral analysis can identify vibration and bearing faults in rotating machinery. Sampling and aliasing matter for robot sensors such as encoders, IMUs and LiDAR. Numerical integration models mechanical systems, while phase portraits help show whether a system loses or conserves energy.

## 11. Conclusion
The experiment demonstrates time-domain and frequency-domain representations of sampled signals, the effect of undersampling, and numerical simulation of a damped nonlinear pendulum. The measured results should be discussed using the values generated by the notebook.

## 12. Viva questions
1. Your spectrum shows a peak at 5 Hz. How do you know it is not a 45 Hz component aliased down?
2. What sets the frequency resolution of a spectrum, and why does sampling faster not improve it?
3. Why must the derivative function take `(t, y)` and not `(y, t)`?
4. What does the area enclosed by a closed phase-portrait trajectory represent?
5. Why does the period diverge as the initial angle approaches 180 degrees?

## Source
CSAI4014 Algorithms for Intelligent Systems and Robotics Laboratory Manual, Experiment 1: Time and frequency domain, Chapter 2, pages 9–15. Numerical implementation uses NumPy, SciPy and Matplotlib.